 # Dungeon'n'Dragons Characterizer #
 Convert your own personality into a real DnD character to enjoy the gAIme!<br>
 <i>Excersise based on Week 1 info</i>


In [ ]:
# base imports, settings and initialization of variables 

import json
import os
import sys

from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

# add week1 directory to sys.path to get scraper module into imports
week1_dir = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..', 'week1'))
sys.path.append(week1_dir)
from scraper import fetch_website_contents, fetch_website_links


# load environment variables
load_dotenv(override=True)
MODEL = "gpt-5-mini"

# assign OPENAI API key from env and check if key exists and is valid
if not (openai_api_key := os.getenv('OPENAI_API_KEY')) or not openai_api_key.startswith("sk-proj-"):
    raise("No API key was found or is invalid (doesn't start sk-proj")
print("API key found and looks good so far!")

openai = OpenAI()



API key found and looks good so far!


Prepare knowledge base (as a list dictionaries) containing information about page (url, type and summary)

In [ ]:
def get_knowledge_base_links(url_base, urlinker_user_prompt):
    '''
    Provide "universal" link selector with help of common system prompt specification - thematic usage depends on request in external user prompt
    '''
    all_urls = fetch_website_links(url_base)

    urlinker_system_prompt = """
    You are provided with a list of links found on a webpage dedicated to certain theme. Based on given intention you are able to decide which of the links 
    would be the most relevant as links necessary for the requested purpose. For example, when you get a company page and the intention is to create a brochure 
    about the company, then you shoudl involve pages like About, Vision and Mission, Annual report or Job opportunities. When you get webpages with soccer rules 
    and the intention is to create a training manual, then you should involve pages like Basic rules, Ball typology, Selecting best football boots. 
    You should respond in JSON as in this example:
    {
        "links": [
            {"type": "about page", "url": "https://full.url/goes/here/about_page", "summary": "short summary of information found on the page"},
            {"type": "game rules page", "url": "https://another.full.url/game_rules", "summary": "short summary of information found on the page"}
        ]
    }
    """

    messages = [
        {"role": "system", "content": urlinker_system_prompt},
        {"role": "user", "content": urlinker_user_prompt + f"\nChoose relevant links from following list: {all_urls}."}
    ]

    print("Consideration of links relevancy started...")
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    knowledge_base_links = json.loads(response.choices[0].message.content)

    print(f"For website {url_base} {len(knowledge_base_links['links'])} of {len(all_urls)} considered as relevant")
    return knowledge_base_links 


Get all relevant links from Dungeon and Dragons rule-site (only links containing information necessary for character creation should be involved).<br>User prompt shoudl be specific.

In [ ]:
dnd_url_base = "https://www.dndbeyond.com/sources/dnd/basic-rules-2014"

urlinker_user_prompt = f"""
You are provided with a webpage links dedicated to a famous table-top role playing game Dungeon and Dragons. The intention is to create a character, 
so the only relevant pages are those dedicated to creation of a game character like choosing a race, personal characteristics, tools or abilities and similar. 
"""

dnd_kb = get_knowledge_base_links(dnd_url_base, urlinker_user_prompt)


Consideration of links relevancy started...
For website https://www.dndbeyond.com/sources/dnd/basic-rules-2014 14 of 195 considered as relevant


Compose DnD character - source information is taken from predefined knowledge base and a personal description of a human player 

In [ ]:

player_websites = [
    fetch_website_contents("https://edwarddonner.com"),
    fetch_website_contents("https://en.wikipedia.org/wiki/Elizabeth_II"),
    fetch_website_contents("https://en.wikipedia.org/wiki/Arnold_Schwarzenegger")
]
personal_summaries = [
    openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Act as helpful snearky assistent."},
            {"role": "user", "content": f"Provide a personal summary from website: {website}"}
        ]
    ) for website in player_websites
]

system_prompt = f"""
Act as a character designer for Dungeon and Dragons player with access to DnD website links summarized in following JSON:\n{json.dumps(dnd_kb)}\n  
Your task is to compose suitable DnD character based on a real characteristics of the human player. For the character's skills do not try to maximize 
the abilities intentionaly but reflect personal information. Do not use Human race for generated character but choose any other suitable race instead.
After the creation of the character write a short explanation why you decided right so"""

for personal_summary in personal_summaries:
    
    user_prompt = f"""Prepare suitable DnD character for a human player described by his or her personal information: {personal_summary}. 
    If possible then use the same style as in previously generated characters."""    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    dnd_character_stream = openai.chat.completions.create(model=MODEL, messages=messages, stream=True)
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in dnd_character_stream:
        response += chunk.choices[0].delta.content or ""
        update_display(Markdown(response), display_id=display_handle.display_id)
    
    # extend user_prompt with "chat history" to keep initial style for all characters
    user_prompt += f"{user_prompt}\nPreviously generated character for keeping the style:\n{response}"




--------------------------------------------



Based on the personal profile of Edward Donner — passionate coder, AI enthusiast, founder and leader in AI startups, educator, electronic music hobbyist, and community engager — here is a carefully crafted Dungeons & Dragons character that reflects his real-life attributes, skillsets, and personality. I deliberately avoided using the Human race, choosing a race that complements Edward's intellect, creativity, and community-focus.

---

### Character Summary

**Name:** Edren Voss  
**Race:** Vedalken  
**Class:** Artificer (Alchemist)  
**Background:** Sage  
**Alignment:** Neutral Good  
**Level:** 1

---

### Race: Vedalken  
*From the Guildmaster’s Guide to Ravnica and further detailed in various DnD sources.*

- **Reasoning:** Vedalken are known for their sharp intellect, analytical approach, and calm demeanor — traits perfectly suited to a skilled coder and AI enthusiast. They possess a natural talent for invention, experimentation, and knowledge, which resonates with Edward’s AI and technological expertise.
- **Traits:**  
  - **Ability Score Increase:** +2 Intelligence, +1 Wisdom — emphasizes intellect and insight over physical prowess.  
  - **Vedalken Dispassion:** Calm and focused mind, great for systematic logic required in coding and research.  
  - **Faculty for Languages and Tools:** Proficient in two extra languages and extra tool proficiencies mirroring Edward’s multidisciplinarity.

---

### Class: Artificer (Alchemist Specialty)  
*Representing Edward’s skill in creation, experimentation, and AI “crafting.”*

- **Reasoning:** Artificer is the class of inventors and innovators using intelligence and magic. The Alchemist subclass, focusing on elixirs and chemical processes, suits Edward’s experimental and scientific mindset as well as his electronic music production (mixing sounds and technology).  
- **Key Abilities:** Intelligence-based spellcasting, tool proficiencies (e.g., tinkering tools), and abilities reflecting crafting and experimentation.

---

### Background: Sage  
- **Reasoning:** A background representing a lifetime of research, education, and passion for knowledge fits Ed’s profile as a teacher with popular courses, a continuous learner, and an expert in his field.  
- **Feature:** Researcher — when you don’t know information, you know where and how to find it, echoing Edward’s deep dive into LLMs and AI resources.  
- **Skills:** Arcana, History — reflect knowledge about complex systems and technologies.

---

### Ability Scores (Point Buy reflecting personality and aptitudes)

| Ability         | Score | Modifier | Justification                                             |
|-----------------|-------|----------|-----------------------------------------------------------|
| Strength        | 8     | -1       | Not focused on physical strength.                         |
| Dexterity       | 13    | +1       | Moderate dexterity helpful for fine manipulation (coding).|
| Constitution    | 12    | +1       | Reasonable resilience to keep going through challenges.  |
| Intelligence   | 16    | +3       | Highest priority, reflecting sharp mind and technical skill. |
| Wisdom          | 14    | +2       | Insightful and thoughtful, good awareness of people and tech.|
| Charisma        | 10    | +0       | Social enough to engage tech communities, but modest.    |

---

### Skills (reflecting Edward’s interests and profession)

- Arcana (Sage Background)  
- History (Sage Background)  
- Investigation (Class choice) — for deep problem-solving  
- Insight (Race and Wisdom) — reading people and systems  
- Persuasion (reflecting community engagement and teaching)  
- Sleight of Hand (reflecting finesse with tools/electronics, music production)  

---

### Starting Equipment

- Alchemist’s supplies  
- Tinker's tools  
- Light crossbow (precision tool)  
- Scholar’s pack  
- An amulet of a stylized AI neural pattern (personal flavor item)  

---

### Personality Traits & Ideals (from Background and Character Flavor)

- **Personality Trait:** I’m inquisitive and always eager to learn new things, especially about how things work.  
- **Ideal:** Knowledge. The path to greatness is through learning and sharing knowledge with others.  
- **Bond:** I owe my skills to those who taught me and want to pass on that legacy.  
- **Flaw:** I sometimes get so deeply focused on a problem that I ignore the world around me.

---

### Explanation of Choices:

- **Vedalken race** was chosen to emphasize intellect, calmness, and a scientific mind without defaulting to the human race. They perfectly match Edward’s cognitive and personality traits.  
- **Artificer (Alchemist)** as a class captures Edward’s creative and experimental nature with technology and AI. The class’s magical tinkering aligns with coding and “creating” from raw materials, symbolic of LLMs and programming.  
- **Sage background** fits a teacher and lifelong learner who produces educational content to help a community.  
- **Ability scores and skills** reflect real strengths in intelligence and wisdom, with moderate dexterity and social aptitude, mirroring collaborative and precision tasks Edward performs.  
- The character avoids extreme optimization, focusing instead on realistic and fitting traits over maximizing combat efficacy.

---

If you want, I can provide the character sheet link with this build on D&D Beyond customized next! Would you like me to do that?


--------------------------------------------



Certainly! Based on the personal summary of Queen Elizabeth II you provided, I’ll craft a suitable Dungeons & Dragons character that captures her essence, qualities, and life story while reflecting her personal traits without using the Human race. I’ll select a race and class that align well with her dignified and resilient nature, along with personality and background choices reflecting her character. Here goes:

---

**Character Name:** Queen Alarielle, The Everlasting Sovereign

**Race:** Elf (High Elf)  
*Reason:* Elves embody longevity, grace, and wisdom—resonating well with Elizabeth II’s long reign and enduring presence. High Elves are noble and traditionally attuned to leadership and culture, matching the regal and dignified persona.

**Class:** Paladin (Oath of Devotion)  
*Reason:* Paladins of the Oath of Devotion stand for duty, honor, and protection of ideals—mirroring Elizabeth’s dedication to duty, stability, and traditional values. Paladins also exude an aura of leadership and inspire loyalty, much like a monarch.

**Background:** Noble  
*Reason:* Noble background fits her royal birth and status, providing skills in History and Persuasion, plus access to noble contacts and the lifestyle fitting a queen.

**Ability Scores:**  
(Using standard array: 15, 14, 13, 12, 10, 8, adjusted to reflect her traits)

- Strength: 13 (strong enough to bear responsibility and command)  
- Dexterity: 12 (graceful and poised)  
- Constitution: 14 (reflecting resilience and longevity)  
- Intelligence: 10 (pragmatic and wise, but not overly academic)  
- Wisdom: 15 (excellent judgment and insight)  
- Charisma: 8 (Though historically charismatic, here shown as dignified but reserved; her influence is through duty rather than overt showmanship)

**Skills:**  
- History (Noble background, high Wisdom)  
- Insight (reflects shrewd understanding of people and events)  
- Persuasion (Noble background; leadership through respect)  
- Religion (reflecting traditional values and public ceremonies)  

**Personality Traits:**  
- Dutiful and disciplined, values tradition and continuity  
- Reserved but deeply compassionate and steady in crisis  

**Ideal:**  
- Responsibility — I must uphold the legacy and stability of my people at all costs.

**Bond:**  
- My realm and subjects are my highest priority; I bear their hopes through all challenges.

**Flaw:**  
- Sometimes unyielding to change, clinging too tightly to tradition in a changing world.

**Equipment:**  
- A finely crafted longsword with royal insignia (symbolic weapon)  
- Full plate armor etched with elven motifs (practical and majestic)  
- Signet ring and noble’s clothing  

---

### Explanation of Choices:

- **Race (High Elf):** Choosing an Elf instead of Human highlights timelessness and an almost ageless aura, fitting for a monarch with a long reign like Elizabeth II. High Elves’ natural nobility aligns with a royal persona without being mundane.

- **Class (Paladin, Oath of Devotion):** Paladins embody service, honor, and a solemn commitment to ideals. Elizabeth II’s reign was marked by duty and stability, which matches well with the Devotion oath’s focus on honesty, courage, compassion, and honor.

- **Background (Noble):** The noble background parallels her real-life royal roots and social standing.

- **Ability Scores:** Emphasizing Wisdom and Constitution reflects her enduring judgment and resilient nature. Charisma is set modestly because while she was influential, her style was more reserved and duty-bound rather than flamboyantly charismatic.

- **Skills and Personality:** Reflects her historic knowledge, diplomatic skill, insight, and religious ties, along with a personality centered on service, duty, and tradition.

This character not only embodies the real qualities of Queen Elizabeth II but also fits well within a D&D campaign setting, offering roleplaying depth and thematic coherence.

---

If you want, I can also prepare a detailed character sheet or build this character using the official [D&D Beyond Character Builder](https://dev.dndbeyond.com/characters/builder). Just let me know!


--------------------------------------------



Here is a suitable Dungeons & Dragons character inspired by Arnold Schwarzenegger’s real-life persona, crafted to reflect his characteristics without choosing Human race and without maximizing abilities artificially:

---

### Character Concept:  
**Name:** Arnok Strongarm  
**Race:** Goliath  
**Class:** Barbarian  
**Background:** Folk Hero

---

### Description and Justification of Choices:

#### Race: Goliath  
- Goliaths are known for their immense physical strength, endurance, and resilience, making them ideal to represent a world-class bodybuilder and action hero physique. Their mountainous tribal background echoes Arnold’s rise from the Austrian Alps region, mirroring rugged origins and a hardy spirit. The racial traits of Goliaths, like Stone’s Endurance (damage reduction) and powerful build, fit well with the persona of a legendary strongman who is both a fighter and survivor.

#### Class: Barbarian  
- The Barbarian class represents physical prowess, raw power, and a larger-than-life presence. It captures the athletic dominance and warrior spirit akin to Schwarzenegger’s bodybuilding and action movie career. Barbarians have rage as a core mechanic giving bursts of strength and combat abilities reflecting the peak of physical capability. The Barbarian’s primal energy symbolizes Arnold’s unstoppable energy during his varied careers.

#### Background: Folk Hero  
- Arnold’s story as a champion and public figure who rose from humble beginnings to global fame suits the Folk Hero background well. This background grants skills and tools to portray someone who is admired by common folk for achievements and service—mirroring Schwarzenegger’s public service as governor and his approachable charisma.

---

### Ability Scores (Reflecting Personal Traits)  
(Assuming Standard Array 15, 14, 13, 12, 10, 8 assigned to attributes)

- Strength: 15 (+2 racial bonus) = 17  
  (Reflects Arnold’s renowned physical strength from bodybuilding)  
- Constitution: 14 (+1 racial bonus) = 15  
  (High endurance from intense training and resilience in his life)  
- Dexterity: 12  
  (Good dexterity to represent athleticism beyond just brute strength)  
- Intelligence: 10  
  (Average intelligence, reflects practical savvy and business sense)  
- Wisdom: 8  
  (Somewhat lower to reflect more focus on physical and practical achievements than introspection)  
- Charisma: 13  
  (Above average; Arnold’s charisma as a performer and politician)

---

### Skills (Reflecting His Real-Life Traits)  
- Athletics (Barbarian class skill, fits strong physicality)  
- Animal Handling (from Folk Hero, suggesting rugged connection to nature and straightforward wisdom)  
- Intimidation (reflects action star persona and commanding presence)  
- Survival (matching background’s rural roots and outdoor resilience)

---

### Traits, Ideals, Bonds, and Flaws  

**Trait:** I’m always willing to help those in need, no matter the personal cost. (Public service and commitment)  
**Ideal:** Responsibility. I’m determined to make a difference with my strength and fame.  
**Bond:** I protect my adopted homeland (California/Goliath tribe) and those who helped me rise.  
**Flaw:** Sometimes my confidence borders on arrogance, and I underestimate subtle problems.

---

### Equipment  
- Greatsword (a big weapon fitting a physically dominant fighter)  
- Explorer’s pack  
- Trophy from a strongman competition (background feature)  
- Traveler’s clothes, a token from his homeland

---

### Summary and Reasoning  
I chose the Goliath race to capture Arnold’s physical might and mountain origins without using Human. The Barbarian class fits his powerful, relentless persona in bodybuilding and action film roles. The Folk Hero background reflects his legendary status and service to common people in politics. Ability scores are assigned to match his physical strength, endurance, and charismatic public image without maximizing unnatural attributes. This character reflects Arnold Schwarzenegger’s multifaceted life in a fantasy D&D context, blending strength, resilience, and leadership.

---

If you want, I can also prepare a full character sheet link referencing DnD Beyond’s builder or help with leveling up!